# 11 — Hierarchical and contextual chunking

## Purpose
This notebook tests whether two frequently recommended approaches actually improve this project's retrieval: **hierarchical parent/child routing** and **LLM-generated contextual chunks**. The existing pipeline is not a naive baseline: Docling already supplies heading paths and the stored text already includes document, entity, period and section metadata.

**Pass criterion:** compare every variant on the same 87 reviewed QRT chunks and six bilingual questions. Report Recall@1/3/5, MRR, first relevant rank, candidates examined and cached latency. A strategy is not adopted merely because one query improves.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
for path in (ROOT / 'backend', ROOT / 'notebooks'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from app.store.artifacts import store
chunks = [c for c in store.chunks if c.document_id == 'foyer_group_qrt_2025']
print(f'Workspace: {ROOT.name}')
print(f'Experiment document chunks: {len(chunks)}')
assert len(chunks) == 87

Workspace: prudential_evidence_lab
Experiment document chunks: 87


## Variants

1. **Current flat:** reviewed runtime chunks with existing metadata.
2. **Structural context:** repeats deterministic document/section/page/type context. This is a negative-control test for context dilution.
3. **Gemini contextual:** prepends one cached, factual chunk-specific sentence generated from the QRT outline and chunk. It is indexed by both BM25 and Gemini.
4. **Hierarchical parent/child:** retrieves section parents first, keeps the best three sections, then ranks their original children so final citations still point to exact pages/cells.

This parent/child experiment is cheaper and simpler than full RAPTOR. It does not recursively cluster or summarize the complete corpus.

In [2]:
REFERENCE_RESULTS = {
    'current_flat': dict(recall_at_1=.5, recall_at_3=2/3, recall_at_5=2/3, mean_reciprocal_rank=.603098, mean_candidates_searched=87, elapsed_seconds=1.003),
    'structural_context': dict(recall_at_1=1/6, recall_at_3=.5, recall_at_5=.5, mean_reciprocal_rank=.383069, mean_candidates_searched=87, elapsed_seconds=.955),
    'gemini_contextual': dict(recall_at_1=.5, recall_at_3=.5, recall_at_5=.5, mean_reciprocal_rank=.561448, mean_candidates_searched=87, elapsed_seconds=.988),
    'hierarchical_parent_child': dict(recall_at_1=2/3, recall_at_3=2/3, recall_at_5=2/3, mean_reciprocal_rank=.693122, mean_candidates_searched=49, elapsed_seconds=.655),
}
for name, row in REFERENCE_RESULTS.items():
    print(f"{name:28} R@1={row['recall_at_1']:.3f} R@3={row['recall_at_3']:.3f} R@5={row['recall_at_5']:.3f} MRR={row['mean_reciprocal_rank']:.3f} candidates={row['mean_candidates_searched']:.1f} latency={row['elapsed_seconds']:.3f}s")

current_flat                 R@1=0.500 R@3=0.667 R@5=0.667 MRR=0.603 candidates=87.0 latency=1.003s
structural_context           R@1=0.167 R@3=0.500 R@5=0.500 MRR=0.383 candidates=87.0 latency=0.955s
gemini_contextual            R@1=0.500 R@3=0.500 R@5=0.500 MRR=0.561 candidates=87.0 latency=0.988s
hierarchical_parent_child    R@1=0.667 R@3=0.667 R@5=0.667 MRR=0.693 candidates=49.0 latency=0.655s


In [3]:
RANKS = {
    'natural_coverage_en': (24, 18, 11, 21),
    'natural_coverage_fr': (13, 10, 9, 9),
    'eligible_own_funds': (2, 2, 1, 1),
    'group_scr': (1, 7, 6, 1),
    'coverage_ratio': (1, 2, 1, 1),
    'row_code': (1, 1, 1, 1),
}
print('query                       current structural contextual hierarchical')
for case, ranks in RANKS.items():
    print(f'{case:28} {ranks[0]:7} {ranks[1]:10} {ranks[2]:11} {ranks[3]:12}')

query                       current structural contextual hierarchical
natural_coverage_en              24         18          11           21
natural_coverage_fr              13         10           9            9
eligible_own_funds                2          2           1            1
group_scr                         1          7           6            1
coverage_ratio                    1          2           1            1
row_code                          1          1           1            1


## Interpretation and decision

- **Hierarchical parent/child routing is the strongest candidate:** MRR rises from 0.603 to 0.693 and mean candidates fall from 87 to 49. Four questions reach rank 1 instead of three.
- **Gemini context helps ambiguous natural phrasing:** English/French natural ranks improve from 24/13 to 11/9 and eligible own funds improves from 2 to 1. However Group SCR falls from 1 to 6, so aggregate Recall@5 decreases.
- **Duplicating structural metadata is harmful:** the chunks already contain this context; adding it again dilutes discriminative QRT terms.
- **Exact codes remain easy:** the row-code query is rank 1 for every strategy. Field-level query reformulation remains necessary for natural questions.

**Decision:** do not replace production chunks with LLM-contextual chunks. Prototype parent/child routing behind a feature flag and expand the benchmark to narrative SFCR/governance questions first. Use Gemini context selectively, offline and cached, only where referents or scope are genuinely ambiguous.

## Reproduce the online run

```powershell
python backend/evals/run_chunking_strategy_eval.py --generate-contexts --dense gemini
```

The 87 contexts and embeddings are cached under `data/processed/chunking_strategy_experiment` and are not committed. The displayed numbers are the stored reference run from 24 August 2026. They characterize one public QRT and six questions, not production performance.